In [4]:
!pip install pyalex tqdm

Defaulting to user installation because normal site-packages is not writeable


In [1]:
#本cell用于测试此处的代码是否可以用于从API获取数据，json部分由ai完成，pyalex部分由我独立阅读文档完成

import json
from pyalex import Works, config #基于OpenAlex API的python库，我在这个项目中用这个库获取所需的数据，并整理成表格形式

config.api_key = "Lw2LtYlvpY9073xqu7hVSx"

query = Works().filter(
    primary_location={"source":{"issn":"0010-938X"}},
    from_publication_date = "2023-01-01",
    language = "en"
).sample(5)

print("请求数据……")
testresult = query.get()

with open('test.json', 'w', encoding='utf-8') as f:
    json.dump(testresult, f, ensure_ascii=False, indent=4)

print(f"成功导入，测试成功")

请求数据……
成功导入，测试成功


In [10]:
#正式获取json形式的数据

import json
from pyalex import Works, config

config.api_key = "Lw2LtYlvpY9073xqu7hVSx"

query = Works().filter(
    primary_location={"source":{"issn":"0010-938X"}},
    from_publication_date = "2023-01-01",
    language = "en"
)

print("请求数据……")

result = []#sample方法参数大于25时会出现分页器问题，这部分代码求助ai解决，在文档中没发现相关解决方案
for i in range(5):
    page_data = query.get(page=i+1, per_page = 100)
    result.extend(page_data)

with open('dataset.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=4)

print(f"导入成功")

请求数据……
导入成功


In [11]:
#检查样本量是否满足设想，json这个库的函数依然是和ai共同完成的
import json

with open('dataset.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"实际抓取的文章数量是: {len(data)}")

实际抓取的文章数量是: 500


In [18]:
import pandas as pd
import json

with open('dataset.json','r',encoding='utf-8') as f:
    raw_data = json.load(f)

rows = []
for item in raw_data:
    rows.append({
        'doi':item.get('doi'),
        'title':item.get('display_name'),
        'year':item.get('publication_year'),
        'citations':item.get('cited_by_count'),
        'is_oa':item.get('open_access',{}).get('is_oa'),
        'type':item.get('type')
    })

df = pd.DataFrame(rows)#由于从OpenAlex获取的数据集为json格式嵌套结构，需要实现一个方法从raw嵌套结构中提取出所需的数据集，因此这里也借助了ai完成部分代码

df.to_csv('dataset.csv',index=False)

print("生成完毕")

生成完毕


In [21]:
print(df.head())
print(df.info())


                                            doi  \
0  https://doi.org/10.1016/j.corsci.2023.111017   
1  https://doi.org/10.1016/j.corsci.2022.110957   
2  https://doi.org/10.1016/j.corsci.2023.111112   
3  https://doi.org/10.1016/j.corsci.2023.111705   
4  https://doi.org/10.1016/j.corsci.2023.111113   

                                               title  year  citations  is_oa  \
0  Metastable pitting corrosion behavior and char...  2023        199   True   
1  Benzothiazole derivatives-based supramolecular...  2023        176  False   
2  On the use of the Langmuir and other adsorptio...  2023        164   True   
3  Modified nano-lignin as a novel biomass-derive...  2023        153  False   
4       Ellingham diagram: A new look at an old tool  2023        136   True   

      type  
0  article  
1  article  
2  article  
3  article  
4  article  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column     Non-Null Count

In [ ]:
# 这部分是api文档中的例程，用于将倒序索引还原为正常的索引文本，没有独立实现能力
import pandas as pd
import json

def rebuild_abstract(inverted_index):
    if not inverted_index or not isinstance(inverted_index, dict):
        return "" 
    
    word_index = []
    for word, pos_list in inverted_index.items():
        for pos in pos_list:
            word_index.append((pos, word))
            
    word_index.sort()
    
    return " ".join([word for pos, word in word_index])

with open('dataset.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

rows = []
for item in raw_data:
    full_text = rebuild_abstract(item.get('abstract_inverted_index'))
    
    rows.append({
        'doi': item.get('doi'),
        'title': item.get('display_name'),
        'year': item.get('publication_year'),
        'citations': item.get('cited_by_count'),
        'is_oa': item.get('open_access', {}).get('is_oa'),
        'abstract': full_text,
    })

df_rich = pd.DataFrame(rows)

df_rich.to_csv('dataset_with_abstract.csv', index=False, encoding='utf-8')

print("转换完成")

转换完成


In [24]:
count = df_rich[df_rich['abstract']!=''].shape[0]
print(count)

62


In [26]:
#发现含摘要的样本量过小，参考文档增加筛选条件

import json
from pyalex import Works, config

config.api_key = "Lw2LtYlvpY9073xqu7hVSx"

query = Works().filter(
    primary_location={"source":{"issn":"0010-938X"}},
    from_publication_date = "2023-01-01",
    language = "en",
    has_abstract=True,
    type="article"
)

print("请求数据……")

result = []#sample方法参数大于25时会出现分页器问题，这部分代码求助ai解决，在文档中没发现相关解决方案
for i in range(5):
    page_data = query.get(page=i+1, per_page = 100)
    result.extend(page_data)

with open('dataset.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=4)

print(f"导入成功")

请求数据……
导入成功


In [28]:
#按照新的筛选条件进行json转换和摘要还原

import pandas as pd
import json

with open('dataset.json','r',encoding='utf-8') as f:
    raw_data = json.load(f)

rows = []
for item in raw_data:
    rows.append({
        'doi':item.get('doi'),
        'title':item.get('display_name'),
        'year':item.get('publication_year'),
        'citations':item.get('cited_by_count'),
        'is_oa':item.get('open_access',{}).get('is_oa'),
        'type':item.get('type')
    })

df = pd.DataFrame(rows)#由于从OpenAlex获取的数据集为json格式嵌套结构，需要实现一个方法从raw嵌套结构中提取出所需的数据集，因此这里也借助了ai完成部分代码

df.to_csv('dataset.csv',index=False)

def rebuild_abstract(inverted_index):
    if not inverted_index or not isinstance(inverted_index, dict):
        return "" 
    
    word_index = []
    for word, pos_list in inverted_index.items():
        for pos in pos_list:
            word_index.append((pos, word))
            
    word_index.sort()
    
    return " ".join([word for pos, word in word_index])

with open('dataset.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

rows = []
for item in raw_data:
    full_text = rebuild_abstract(item.get('abstract_inverted_index'))
    
    rows.append({
        'doi': item.get('doi'),
        'title': item.get('display_name'),
        'year': item.get('publication_year'),
        'citations': item.get('cited_by_count'),
        'is_oa': item.get('open_access', {}).get('is_oa'),
        'abstract': full_text,
    })

df_rich = pd.DataFrame(rows)

df_rich.to_csv('dataset_with_abstract.csv', index=False, encoding='utf-8')

print("处理结束")

处理结束


In [4]:
import pandas as pd
df = pd.read_csv('dataset_with_abstract.csv')
print(df.head())

                                            doi  \
0  https://doi.org/10.1016/j.corsci.2023.111017   
1  https://doi.org/10.1016/j.corsci.2023.111112   
2  https://doi.org/10.1016/j.corsci.2023.111025   
3  https://doi.org/10.1016/j.corsci.2023.111296   
4  https://doi.org/10.1016/j.corsci.2023.111591   

                                               title  year  citations  is_oa  \
0  Metastable pitting corrosion behavior and char...  2023        199   True   
1  On the use of the Langmuir and other adsorptio...  2023        164   True   
2  Experimental comparison of gaseous and electro...  2023        118   True   
3  Synergistic inhibition effect of Mikania micra...  2023        104   True   
4  Study on the oxidation behavior of a novel the...  2023        100   True   

                                            abstract  
0  This work investigated the metastable pitting ...  
1  In corrosion inhibition studies, the standard ...  
2  Herein, hydrogen uptake and diffusivity in X

In [8]:
df_abs = df["abstract"]
df_hedge = pd.read_csv('hedge_data.txt',header=None)
print(df_abs.head(),"\n",df_hedge.head())

0    This work investigated the metastable pitting ...
1    In corrosion inhibition studies, the standard ...
2    Herein, hydrogen uptake and diffusivity in X65...
3    The synergistic inhibition effect of Mikania m...
4    A novel thermal barrier coating system with a ...
Name: abstract, dtype: object 
            0
0    largely
1  generally
2      often
3     rarely
4  sometimes


In [16]:
def count_hedges(text):
  if pd.isna(text):
      return 0
  text = text.lower()
  return sum(df_hedge[0].str.lower().apply(lambda h: h in text))

df['hedge'] = df['abstract'].apply(count_hedges)
print(df['hedge'].head(),"\n",df.head())
df.to_csv("dataset_with_hedge.csv")

0    2
1    6
2    0
3    2
4    1
Name: hedge, dtype: int64 
                                             doi  \
0  https://doi.org/10.1016/j.corsci.2023.111017   
1  https://doi.org/10.1016/j.corsci.2023.111112   
2  https://doi.org/10.1016/j.corsci.2023.111025   
3  https://doi.org/10.1016/j.corsci.2023.111296   
4  https://doi.org/10.1016/j.corsci.2023.111591   

                                               title  year  citations  is_oa  \
0  Metastable pitting corrosion behavior and char...  2023        199   True   
1  On the use of the Langmuir and other adsorptio...  2023        164   True   
2  Experimental comparison of gaseous and electro...  2023        118   True   
3  Synergistic inhibition effect of Mikania micra...  2023        104   True   
4  Study on the oxidation behavior of a novel the...  2023        100   True   

                                            abstract  hedge  
0  This work investigated the metastable pitting ...      2  
1  In corrosion inhibitio

In [1]:
import pandas as pd
import numpy as np
from pandas import DataFrame
df=pd.read_csv('dataset_with_abstract.csv')
#print(df.head())
def fun1(text):
    ccount_1 = len(text)
    pun=',.\'"!-%~'
    punc_count = 0
    text_nopunc = ""
    for char in text:
        if char in pun:
            punc_count+=1
        else:
            text_nopunc+=char
    ccount_2=len(text_nopunc)
    space_c=text_nopunc.count(' ')
    word_c=space_c+1
    if word_c>0:
        mean_word_length = ccount_2/word_c
    else:
        mean_word_length = 0
    return pd.Series([ccount_1, punc_count, ccount_2, space_c, word_c, mean_word_length])
new_cols = ['Original_CCount', 'PunCount', 'CCount_No_Punc', 'Space_Count', 'Word_Count', 'Mean_Word_Length']
df[new_cols] = df['abstract'].apply(fun1)
print(df[['title', 'Word_Count', 'Mean_Word_Length']].head())
dataset_with_features_Meanwordlength='dataset_with_features.csv'
df.to_csv(dataset_with_features_Meanwordlength, index=False)

                                               title  Word_Count  \
0  Metastable pitting corrosion behavior and char...       167.0   
1  On the use of the Langmuir and other adsorptio...       131.0   
2  Experimental comparison of gaseous and electro...        77.0   
3  Synergistic inhibition effect of Mikania micra...        97.0   
4  Study on the oxidation behavior of a novel the...        95.0   

   Mean_Word_Length  
0          6.413174  
1          6.847328  
2          7.714286  
3          6.783505  
4          7.126316  
